# San Francisco-i Bűnügyi Predikciós Rendszer - V6
Ez a verzió a predikcióra fókuszál: tartalmazza a manuális szimulátort és egy új, automatikus 'Következő Napi Előrejelző' modult.

In [1]:
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from sklearn.preprocessing import LabelEncoder
import warnings
from datetime import timedelta
warnings.filterwarnings('ignore')

## 1. Adatok betöltése és előkészítése

In [2]:
# Adatok betöltése
df = pd.read_csv('Map-Crime_Incidents-Previous_Three_Months.csv')

# Alapvető tisztítás és típuskonverzió
df['IncidntNum'] = df['IncidntNum'].fillna(0).astype(int)
df['Date'] = pd.to_datetime(df['Date'])
df['Hour'] = pd.to_datetime(df['Time'], format='%H:%M').dt.hour

# Napszak kategóriák
def categorize_time(hour):
    if 0 <= hour < 6: return 'Hajnal'
    elif 6 <= hour < 12: return 'Reggel'
    elif 12 <= hour < 18: return 'Délután'
    else: return 'Éjszaka'

df['Time_of_Day'] = df['Hour'].apply(categorize_time)

# Érvénytelen koordináták szűrése
df = df[(df['X'] < -120) & (df['Y'] > 30)]

# Ritka kategóriák szűrése a modell stabilitása érdekében
cat_counts = df['Category'].value_counts()
valid_cats = cat_counts[cat_counts >= 15].index
df_ml = df[df['Category'].isin(valid_cats)].copy()

# Label Encoding
le_cat = LabelEncoder()
le_day = LabelEncoder()
le_dist = LabelEncoder()
le_tod = LabelEncoder()

y = le_cat.fit_transform(df_ml['Category'])
df_ml['Day_Encoded'] = le_day.fit_transform(df_ml['DayOfWeek'])
df_ml['District_Encoded'] = le_dist.fit_transform(df_ml['PdDistrict'])
df_ml['TOD_Encoded'] = le_tod.fit_transform(df_ml['Time_of_Day'])

features = ['Day_Encoded', 'District_Encoded', 'Hour', 'X', 'Y', 'TOD_Encoded']
X = df_ml[features]

# Modell tanítása
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
clf = xgb.XGBClassifier(n_estimators=100, max_depth=6, learning_rate=0.1, random_state=42, objective='multi:softprob', eval_metric='mlogloss')
clf.fit(X_train, y_train)
print("A predikciós modell készen áll.")

A predikciós modell készen áll.


## 2. Manuális Bűnügyi Szimulátor
Ebben a részben tetszőleges paramétereket adhatsz meg a modellnek.

In [3]:
def predict_crime_probability(hour, day_name, x, y, top_n=3):
    try:
        day_encoded = le_day.transform([day_name])[0]
    except:
        day_encoded = 0
    
    # Körzet keresése (koordináták alapján a legközelebbi körzet központját rendeljük hozzá)
    # Most az egyszerűség kedvéért a modell tanításakor használt körzet kódolást használjuk
    tod_name = categorize_time(hour)
    tod_encoded = le_tod.transform([tod_name])[0]
    
    # Itt fix körzetet használunk (Southern), de a következő napi modellnél dinamikus lesz
    district_encoded = le_dist.transform(['SOUTHERN'])[0]
    
    input_data = pd.DataFrame([[day_encoded, district_encoded, hour, x, y, tod_encoded]], columns=features)
    probabilities = clf.predict_proba(input_data)[0]
    top_indices = probabilities.argsort()[-top_n:][::-1]
    
    print(f"--- MANUÁLIS SZIMULÁCIÓ ---")
    print(f"IDŐPONT: {day_name}, {hour}:00 ({tod_name}) | HELYSZÍN: É:{y}, K:{x}")
    for idx in top_indices:
        print(f"  > {le_cat.classes_[idx]}: {probabilities[idx]*100:.2f}%")

# Példa: Péntek 22:00
predict_crime_probability(22, 'Friday', -122.41, 37.78)

--- MANUÁLIS SZIMULÁCIÓ ---
IDŐPONT: Friday, 22:00 (Éjszaka) | HELYSZÍN: É:37.78, K:-122.41
  > LARCENY/THEFT: 32.48%
  > ASSAULT: 13.20%
  > OTHER OFFENSES: 11.92%


## 3. Automatikus Következő Napi Előrejelzés
A rendszer megkeresi az utolsó ismert dátumot a CSV-ben, kiszámítja a következő (n+1) napot, és megkeresi a legnagyobb valószínűségű bűncselekményt.

In [4]:
# 1. Utolsó dátum és a következő nap meghatározása
last_date = df['Date'].max()
next_day = last_date + timedelta(days=1)
next_day_name = next_day.strftime('%A')
print(f"Elemzés alatt álló jövőbeli dátum: {next_day.date()} ({next_day_name})")

# 2. Előrejelzési háló (Grid) létrehozása
# Minden körzet központi koordinátáját és minden fő napszakot vizsgálunk
districts = df_ml.groupby('PdDistrict').agg({'X': 'mean', 'Y': 'mean', 'District_Encoded': 'first'}).reset_index()
hours_to_test = [3, 9, 15, 21] # Hajnal, Reggel, Délután, Éjszaka reprezentatív órái

forecast_results = []

for _, district_row in districts.iterrows():
    for hr in hours_to_test:
        day_enc = le_day.transform([next_day_name])[0]
        tod_enc = le_tod.transform([categorize_time(hr)])[0]
        
        input_row = pd.DataFrame([[
            day_enc, 
            district_row['District_Encoded'], 
            hr, 
            district_row['X'], 
            district_row['Y'], 
            tod_enc
        ]], columns=features)
        
        probs = clf.predict_proba(input_row)[0]
        max_idx = probs.argmax()
        max_prob = probs[max_idx]
        crime_cat = le_cat.classes_[max_idx]
        
        forecast_results.append({
            'District': district_row['PdDistrict'],
            'Hour': hr,
            'TimeOfDay': categorize_time(hr),
            'Crime': crime_cat,
            'Probability': max_prob,
            'X': district_row['X'],
            'Y': district_row['Y']
        })

# 3. A legmagasabb valószínűségű esemény kiválasztása
forecast_df = pd.DataFrame(forecast_results)
top_event = forecast_df.loc[forecast_df['Probability'].idxmax()]

print("-" * 50)
print(f"--- HIVATALOS ELŐREJELZÉS A KÖVETKEZŐ NAPRA ({next_day.date()}) ---")
print(f"A legnagyobb eséllyel bekövetkező esemény:")
print(f"DÁTUM: {next_day.date()} ({next_day_name})")
print(f"NAPSZAK: {top_event['TimeOfDay']} ({top_event['Hour']}:00 óra körül)")
print(f"HELYSZÍN: {top_event['District']} körzet (Koordináták: {top_event['Y']:.4f}, {top_event['X']:.4f})")
print(f"BŰNCSELEKMÉNY: {top_event['Crime']}")
print(f"VALÓSZÍNŰSÉG: {top_event['Probability']*100:.2f}%")
print("-" * 50)

Elemzés alatt álló jövőbeli dátum: 2014-09-01 (Monday)
--------------------------------------------------
--- HIVATALOS ELŐREJELZÉS A KÖVETKEZŐ NAPRA (2014-09-01) ---
A legnagyobb eséllyel bekövetkező esemény:
DÁTUM: 2014-09-01 (Monday)
NAPSZAK: Éjszaka (21:00 óra körül)
HELYSZÍN: NORTHERN körzet (Koordináták: 37.7864, -122.4267)
BŰNCSELEKMÉNY: LARCENY/THEFT
VALÓSZÍNŰSÉG: 41.06%
--------------------------------------------------
